In [2]:
# torch is the fundamental numerical computing library
# It gives us the 'tensor' object — the GPU/CPU-aware equivalent of a numpy array
# Every weight matrix in the model is a torch.Tensor
import torch

# transformers is HuggingFace's library
# AutoModelForCausalLM: a smart loader that reads config.json,
#   figures out the right model class (Phi3ForCausalLM in our case),
#   builds the module tree, and fills it with weights from the .safetensors file
# AutoTokenizer: loads the vocabulary and handles text → token ID conversion
from transformers import AutoModelForCausalLM, AutoTokenizer

/home/codespace/.local/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
model_id = "microsoft/Phi-4-mini-instruct"

print("Loading tokenizer...")
tokenizer = AutoTokenizer.from_pretrained(
    model_id,
    trust_remote_code=True   # Phi-4-mini uses custom code in the repo, this allows it to run
)

print("Loading model... (this will take 1-3 minutes and ~8GB RAM)")
model = AutoModelForCausalLM.from_pretrained(
    model_id,
    torch_dtype=torch.bfloat16,   # load weights in BF16, matching the original format
    device_map="cpu",              # explicitly put everything on CPU — no GPU assumed
    trust_remote_code=True
)

print("Model loaded.")
print(f"Total parameters: {sum(p.numel() for p in model.parameters()):,}")


Loading tokenizer...


Loading model... (this will take 1-3 minutes and ~8GB RAM)


Loading checkpoint shards: 100%|██████████| 2/2 [00:09<00:00,  4.75s/it]


Model loaded.
Total parameters: 3,836,021,760


In [4]:
# model.model  →  the Phi3Model (inner body, excluding lm_head)
# .layers      →  the ModuleList of 32 DecoderLayers
# [0]          →  Layer 0 specifically (index 0 of the list)
# .self_attn   →  the Phi3Attention sub-module inside that layer
# .qkv_proj   →  the Linear layer (contains .weight and optionally .bias)
# .weight      →  the actual tensor of numbers — shape [5120, 3072]
layer0_qkv = model.model.layers[0].self_attn.qkv_proj.weight

print(f"Type:   {type(layer0_qkv)}")
print(f"Shape:  {layer0_qkv.shape}")
print(f"Dtype:  {layer0_qkv.dtype}")
print(f"Device: {layer0_qkv.device}")


# .detach() — creates a copy disconnected from PyTorch's computation graph
#   We do this because we're going to do manual math on this tensor
#   Without detach(), PyTorch would try to track every operation for gradient computation
#   which wastes memory and isn't needed since we're not training
#
# .float() — converts from BF16 to FP32 for our inspection math
#   BF16 arithmetic can have small rounding surprises during computation
#   FP32 gives us clean, predictable numbers for analysis
#   We are NOT changing the model's stored weights — this is a local copy
w = layer0_qkv.detach().float()

# These four numbers are your baseline — you will reference them constantly
print(f"Min value:       {w.min().item():.6f}")
print(f"Max value:       {w.max().item():.6f}")
print(f"Max absolute:    {w.abs().max().item():.6f}")
print(f"Mean absolute:   {w.abs().mean().item():.6f}")

Type:   <class 'torch.nn.parameter.Parameter'>
Shape:  torch.Size([5120, 3072])
Dtype:  torch.bfloat16
Device: cpu


Min value:       -1.593750
Max value:       1.460938
Max absolute:    1.593750
Mean absolute:   0.026782


Now read these numbers carefully before we write anything.

The mean absolute value is 0.026782. The max absolute value is 1.593750. That ratio — 1.593750 / 0.026782 — is approximately 59×. The single largest value in this matrix is 59 times bigger than the average value. That is your first real signal. The matrix is not uniformly distributed — there is at least one value dramatically larger than the typical value.

The scale calculation you were supposed to do yourself:

Scale = max_abs / 127 = 1.593750 / 127 = 0.012549

This means each "step" between adjacent INT8 values represents 0.012549 in BF16 terms. Now think about what this does to the average value: mean_abs / scale = 0.026782 / 0.012549 ≈ 2.13 integer slots. The average weight in this matrix maps to slot 2 out of 127. The vast majority of values in this matrix are being crushed into the bottom few integer slots, while the scale is stretched wide to accommodate that one large outlier at 1.593750. This is the crushing effect from your screenshot, happening in your actual numbers, right now.

You now have the motivation for per-channel scaling in your own data, not as something you read about.

so  max is the highest weight in the matrix  min is the lowest
max absolute is probably the highest absolute value of the entire distribution which here is the 1.593750 but I dunno what mean absolute signifies

also I understand that the max value to be represented divided by the toatal range gives us the slot size so 1.593750 / 127 is the slot size and then  to find the slot where number belongs we take ratio of actual value/slot size  I dunno why do u ask me to divide max value to be represented by the mean and what does it being 59x signify I know that dividing the mean by slot size gives avg of the slots  which is 2.13 but the real size is the actual number divided by the slot size  but  how do I identify where does the outlier originate also we didn't really pass any tokens so there is no product are we just inspecting outliers in post trained weights ?

 lets go step by step

What does mean absolute signify?

Min and max tell you the two extreme endpoints of the distribution — the worst cases. But they tell you nothing about where most values actually live. Mean absolute is the average size of a value ignoring its sign — it tells you what a "typical" value looks like.

Think of it this way: if a class of 30 students scores between 0 and 100, and the mean is 47, you know most students are clustered around the middle. If the mean is 3 but the max is 100, you know almost everyone scored very low and one person aced it. That's exactly your situation — mean_abs of 0.026 but max_abs of 1.59 tells you the vast majority of values are tiny, and a small number of values are dramatically larger.

Why does the 59× ratio matter?

You're right that the slot size is 1.593750 / 127 = 0.012549. Now think about what happens to a typical value — say 0.026 (the mean). Its slot number is 0.026 / 0.012549 ≈ 2. Slot 2 out of 127. A value that's twice the mean — 0.052 — maps to slot 4. A value that's three times the mean maps to slot 6.

So roughly 95% of your 15.7 million values are being packed into slots 0 through 10, while slots 11 through 127 sit mostly empty, reserved for the rare large values. You have 127 available slots and you're effectively using about 10 of them for the bulk of the data. That's the crushing — not a dramatic collapse, but a quiet one where almost all precision gets wasted.

The 59× ratio is just a quick way to see this without computing the histogram yet — if max is 59× the mean, the scale is set 59× wider than the typical value needs, which means typical values get 1/59th of the available precision.

Why divide mean by slot size specifically?

It's not a special formula — it's just asking "where does the average value land on the integer grid?" It's the same calculation as any value's slot, just applied to the mean so you get a feel for where the bulk of the distribution sits relative to the 127 available slots. You could do it for any value. Try: where does the value 0.5 land? Where does -0.8 land? Same formula each time.

How do you identify where an outlier originates?

Right now you can't — and you're not supposed to yet. What you're doing now is purely inspecting the weights of the pre-trained model. These weight values were shaped by training on 5 trillion tokens, and the outliers you're seeing in the weight matrix are a consequence of what the model learned, not something you can trace back to a specific cause just by looking at the numbers.

The outliers in activations (the values that flow through the model when you actually run text) are a separate thing — those are what AWQ's calibration step captures, and that's Stage 3 of your curriculum. You're in Stage 1 right now. At Stage 3, you will pass real tokens through the model and observe which input channels to qkv_proj consistently have large values — those are the activation outliers. The weight outliers you're seeing now are related but not identical.

Your last question is the most important one and you got it exactly right.

Yes — you have not passed any tokens. There is no forward pass happening. You are looking directly at the frozen weight matrix as it sits on disk. The 15.7 million numbers you inspected are the numbers Microsoft's training process arrived at after 5 trillion tokens of gradient updates. You are asking: "given these specific stored numbers, what happens when I try to round them to INT8?" That question is answerable purely from the weights themselves, without running any text through the model.

Activation outliers — the spikes caused by punctuation and BOS tokens — only appear when you run a forward pass. That's Stage 3. Right now in Stage 1 you're asking a simpler question: "how bad is naive rounding for this one weight matrix, ignoring activations entirely?" That's the baseline every more sophisticated method needs to beat.

In [5]:
# Same histogram, but sorted to show the most populated buckets first
# This lets us see where the bulk of values actually live
hist = torch.histc(w, bins=50, min=w.min().item(), max=w.max().item())
bucket_width = (w.max().item() - w.min().item()) / 50

buckets = []
for i, count in enumerate(hist):
    bucket_left = w.min().item() + i * bucket_width
    count_int = int(count.item())
    buckets.append((count_int, bucket_left, bucket_left + bucket_width))

# Sort by count descending, show top 15
buckets.sort(reverse=True)
print("\nTop 15 most populated buckets:")
print(f"{'Range':>25}  {'Count':>10}  {'% of total':>10}")
print("-" * 60)
total = w.numel()
for count, left, right in buckets[:15]:
    pct = 100.0 * count / total
    print(f"  [{left:+.3f} to {right:+.3f}]  {count:>10,}  {pct:>9.2f}%")

# Also print these specific summary stats
print(f"\nTotal values in matrix: {total:,}")
print(f"Values between -0.1 and +0.1: {((w > -0.1) & (w < 0.1)).sum().item():,}")
print(f"That is: {100.0 * ((w > -0.1) & (w < 0.1)).sum().item() / total:.1f}% of all values")
print(f"Values beyond ±0.5: {((w.abs() > 0.5)).sum().item():,}")
print(f"That is: {100.0 * (w.abs() > 0.5).sum().item() / total:.1f}% of all values")


Top 15 most populated buckets:
                    Range       Count  % of total
------------------------------------------------------------
  [-0.005 to +0.056]   8,227,407      52.31%
  [-0.066 to -0.005]   6,048,842      38.46%
  [+0.056 to +0.117]     774,529       4.92%
  [-0.127 to -0.066]     498,270       3.17%
  [+0.117 to +0.178]      81,201       0.52%
  [-0.189 to -0.127]      59,178       0.38%
  [+0.178 to +0.239]      14,805       0.09%
  [-0.250 to -0.189]      11,252       0.07%
  [+0.239 to +0.300]       4,102       0.03%
  [-0.311 to -0.250]       3,444       0.02%
  [+0.300 to +0.361]       1,408       0.01%
  [-0.372 to -0.311]       1,237       0.01%
  [+0.361 to +0.422]         662       0.00%
  [-0.433 to -0.372]         551       0.00%
  [+0.422 to +0.483]         351       0.00%

Total values in matrix: 15,728,640
Values between -0.1 and +0.1: 15,375,742
That is: 97.8% of all values
Values beyond ±0.5: 1,016
That is: 0.0% of all values


97.8% of 15.7 million values live between -0.1 and +0.1.

Only 1,016 values — out of 15,728,640 — exceed ±0.5.

And yet — those 1,016 values are setting the scale for all 15,728,640. Because naive quantization takes the single largest absolute value (1.593750) and uses it to define the slot size for the entire matrix.



Let's make this concrete with your actual numbers:

Scale (naive, whole matrix) = 1.593750 / 127 = 0.012549 per slot

A value of 0.050 (very typical, inside the fat middle bucket) maps to:
    0.050 / 0.012549 = 3.98 → rounds to slot 4

A value of 0.030 maps to slot 2.
A value of 0.010 maps to slot 1.
A value of 0.005 maps to slot 0.
A value of -0.005 maps to slot 0.

Everything between -0.006 and +0.006 maps to slot 0.




That -0.005 to +0.056 bucket alone contains 8.2 million values — 52% of the entire matrix — and they're all being compressed into roughly slots 0 through 4. Five slots. Out of 127 available.

The 1,016 outlier values beyond ±0.5 are using slots 40 through 127. Eighty-seven slots. For 0.006% of values.

This is not a small inefficiency. This is the entire problem you are solving.

so how do I visualise the matrix  as rows and columns  I mean 
 [-0.005 to +0.056]   8,227,407      52.31%   [-0.066 to -0.005]   6,048,842      38.46%
these are just the ranges and the valuecounts  and percentages lying in the ranges 

there is no insight on how the weights are placesd in the matrix  how I I know the trend whether the weights are outliers in a trend of columns or rows  I mean how do I imagine the  matrix positional trends I'll need that insight   to take the decision of whether row scaling or column scaling is better?

why did the company even release such a defective model its pretrained yet has outliers even before  the tokens are passed how can it ever give good response

On the "defective model" question — this is a fundamental misconception worth correcting properly.

The model is not defective. It gives excellent responses in BF16. The outliers are not bugs — they are a direct consequence of the model learning to be good.

Here's what actually happens during training. The model is shown trillions of examples. At some point, training discovered that putting a very large weight value in a specific channel of a specific layer dramatically reduced prediction error. Gradient descent — the optimization process — has no constraint saying "keep all weights small." It only cares about one thing: minimize the loss. If a large weight value helps, it stays large. The model doesn't know or care that you'll try to quantize it later.

The outliers you're seeing are features, not bugs. They represent something the model learned was important — a specific direction in weight space that carries disproportionate information. This is why naive quantization degrades quality: you're not rounding unimportant noise, you're rounding something the model specifically learned to make large because it mattered.

This is also why AWQ exists. It was invented precisely because researchers noticed that post-trained models universally have this property — not just Phi-4-mini, but GPT, LLaMA, Mistral, every model above ~6.7B parameters. It's not a Microsoft problem. It's a property of how large neural networks train.

In [6]:
# w is shape [5120, 3072]
# axis=1 means "compute across the 3072 columns, keeping rows separate"
# So row_max_abs[i] = the largest absolute value anywhere in row i
# Result shape: [5120] — one number per row
row_max_abs = w.abs().max(dim=1).values

# Same idea but for columns
# axis=0 means "compute across the 5120 rows, keeping columns separate"  
# col_max_abs[j] = the largest absolute value anywhere in column j
# Result shape: [3072] — one number per column
col_max_abs = w.abs().max(dim=0).values

print("=== ROW ANALYSIS (5120 rows) ===")
print(f"Median row max-abs:  {row_max_abs.median().item():.6f}")
print(f"Mean row max-abs:    {row_max_abs.mean().item():.6f}")
print(f"Max row max-abs:     {row_max_abs.max().item():.6f}")
print(f"Rows where max-abs > 0.5:  {(row_max_abs > 0.5).sum().item()}")
print(f"Rows where max-abs > 1.0:  {(row_max_abs > 1.0).sum().item()}")

print("\n=== COLUMN ANALYSIS (3072 columns) ===")
print(f"Median col max-abs:  {col_max_abs.median().item():.6f}")
print(f"Mean col max-abs:    {col_max_abs.mean().item():.6f}")
print(f"Max col max-abs:     {col_max_abs.max().item():.6f}")
print(f"Cols where max-abs > 0.5:  {(col_max_abs > 0.5).sum().item()}")
print(f"Cols where max-abs > 1.0:  {(col_max_abs > 1.0).sum().item()}")

print("\n=== TOP 10 WORST ROWS (highest max-abs) ===")
top_rows = row_max_abs.topk(10)
for rank, (val, idx) in enumerate(zip(top_rows.values, top_rows.indices)):
    print(f"  Rank {rank+1}: Row {idx.item():>5}  max-abs = {val.item():.6f}")

print("\n=== TOP 10 WORST COLUMNS (highest max-abs) ===")
top_cols = col_max_abs.topk(10)
for rank, (val, idx) in enumerate(zip(top_cols.values, top_cols.indices)):
    print(f"  Rank {rank+1}: Col {idx.item():>5}  max-abs = {val.item():.6f}")

=== ROW ANALYSIS (5120 rows) ===
Median row max-abs:  0.163086
Mean row max-abs:    0.201939
Max row max-abs:     1.593750
Rows where max-abs > 0.5:  195
Rows where max-abs > 1.0:  20

=== COLUMN ANALYSIS (3072 columns) ===
Median col max-abs:  0.231445
Mean col max-abs:    0.252314
Max col max-abs:     1.593750
Cols where max-abs > 0.5:  61
Cols where max-abs > 1.0:  16

=== TOP 10 WORST ROWS (highest max-abs) ===
  Rank 1: Row  2535  max-abs = 1.593750
  Rank 2: Row  2517  max-abs = 1.398438
  Rank 3: Row  3376  max-abs = 1.375000
  Rank 4: Row  2528  max-abs = 1.320312
  Rank 5: Row  1343  max-abs = 1.273438
  Rank 6: Row  2559  max-abs = 1.265625
  Rank 7: Row  4055  max-abs = 1.218750
  Rank 8: Row  2538  max-abs = 1.218750
  Rank 9: Row  3798  max-abs = 1.210938
  Rank 10: Row  3827  max-abs = 1.203125

=== TOP 10 WORST COLUMNS (highest max-abs) ===
  Rank 1: Col  1541  max-abs = 1.593750
  Rank 2: Col  2412  max-abs = 1.460938
  Rank 3: Col  2293  max-abs = 1.429688
  Rank 4: Co

In [7]:
# The matrix w has shape [5120, 3072]
# Think of it as a spreadsheet:
# - 5120 rows (row 0, row 1, row 2, ... row 5119)
# - 3072 columns (col 0, col 1, col 2, ... col 3071)
# Each cell contains one weight value

# ---------------------------------------------------
# UNDERSTANDING dim=1 (collapsing across columns)
# ---------------------------------------------------
# For ROW 0 specifically, the operation is:
#   look at all 3072 values in row 0: [0.021, -0.003, 0.147, ..., -0.089]
#   find the largest absolute value among those 3072 numbers
#   store that one number as row_max_abs[0]
#
# For ROW 1:
#   look at all 3072 values in row 1
#   find the largest absolute value
#   store as row_max_abs[1]
#
# This repeats for all 5120 rows.
# Result: row_max_abs has shape [5120]
#         row_max_abs[i] = "what is the worst value anywhere in row i?"

row_max_abs = w.abs().max(dim=1).values
# dim=1 means "collapse the column dimension"
# i.e. for each row, reduce its 3072 values down to 1 number (the max abs)

# ---------------------------------------------------
# UNDERSTANDING dim=0 (collapsing across rows)
# ---------------------------------------------------
# For COLUMN 0:
#   look at all 5120 values in column 0: [0.004, -0.021, 0.003, ..., 0.011]
#   find the largest absolute value among those 5120 numbers
#   store as col_max_abs[0]
#
# For COLUMN 1:
#   look at all 5120 values in column 1
#   find the largest absolute value
#   store as col_max_abs[1]
#
# Result: col_max_abs has shape [3072]
#         col_max_abs[j] = "what is the worst value anywhere in column j?"

col_max_abs = w.abs().max(dim=0).values
# dim=0 means "collapse the row dimension"
# i.e. for each column, reduce its 5120 values down to 1 number (the max abs)

In [8]:
# Find the single cell with the largest absolute value in the entire matrix
# unravel_index converts a flat index back into (row, col) coordinates
flat_idx = w.abs().argmax()          # position in the flattened 15.7M list
row_idx, col_idx = divmod(flat_idx.item(), w.shape[1])

print(f"Largest absolute value: {w.abs().max().item():.6f}")
print(f"Located at row {row_idx}, column {col_idx}")
print(f"Actual value there: {w[row_idx, col_idx].item():.6f}")

# Now look at the full row that contains this outlier
outlier_row = w[row_idx, :]          # shape [3072] — all 3072 values in that row
print(f"\nRow {row_idx} statistics:")
print(f"  Min:      {outlier_row.min().item():.6f}")
print(f"  Max:      {outlier_row.max().item():.6f}")
print(f"  Mean abs: {outlier_row.abs().mean().item():.6f}")
print(f"  Max abs:  {outlier_row.abs().max().item():.6f}")

# Compare against a normal row — pick row 0 which ranked nowhere near top 10
normal_row = w[0, :]
print(f"\nRow 0 (normal) statistics:")
print(f"  Min:      {normal_row.min().item():.6f}")
print(f"  Max:      {normal_row.max().item():.6f}")
print(f"  Mean abs: {normal_row.abs().mean().item():.6f}")
print(f"  Max abs:  {normal_row.abs().max().item():.6f}")

# Now look at the full column that contains this outlier
outlier_col = w[:, col_idx]          # shape [5120] — all 5120 values in that column
print(f"\nColumn {col_idx} statistics:")
print(f"  Min:      {outlier_col.min().item():.6f}")
print(f"  Max:      {outlier_col.max().item():.6f}")
print(f"  Mean abs: {outlier_col.abs().mean().item():.6f}")
print(f"  Max abs:  {outlier_col.abs().max().item():.6f}")

# Compare against a normal column
normal_col = w[:, 0]
print(f"\nColumn 0 (normal) statistics:")
print(f"  Min:      {normal_col.min().item():.6f}")
print(f"  Max:      {normal_col.max().item():.6f}")
print(f"  Mean abs: {normal_col.abs().mean().item():.6f}")
print(f"  Max abs:  {normal_col.abs().max().item():.6f}")

Largest absolute value: 1.593750
Located at row 2535, column 1541
Actual value there: -1.593750

Row 2535 statistics:
  Min:      -1.593750
  Max:      1.460938
  Mean abs: 0.148997
  Max abs:  1.593750

Row 0 (normal) statistics:
  Min:      -0.155273
  Max:      0.151367
  Mean abs: 0.016973
  Max abs:  0.155273

Column 1541 statistics:
  Min:      -1.593750
  Max:      1.335938
  Mean abs: 0.086567
  Max abs:  1.593750

Column 0 (normal) statistics:
  Min:      -0.231445
  Max:      0.242188
  Mean abs: 0.028037
  Max abs:  0.242188
